# **AnnData 数据结构详解**

AnnData（Annotated Data）是单细胞RNA测序分析的核心数据结构，专门为高维注释数据设计。以下是完整的组成部分详解：

## **核心组成部分**

### **1. X：核心表达矩阵**
```python
adata.X  # 主数据矩阵
```
- **类型**：通常是`scipy.sparse.csr_matrix`（稀疏矩阵）或`numpy.ndarray`
- **形状**：`(n_obs × n_var)` - （细胞数 × 基因数）
- **内容**：基因表达值（原始计数、归一化值等）
- **特点**：
  ```python
  print(type(adata.X))        # <class 'scipy.sparse.csr_matrix'>
  print(adata.X.shape)        # (5000, 20000) - 5000个细胞，20000个基因
  print(adata.X.nnz)          # 非零元素数量
  ```

### **2. obs：细胞注释数据**
```python
adata.obs  # 观测（细胞）的注释信息
```
- **类型**：`pandas.DataFrame`
- **形状**：`(n_obs, n_obs_columns)`
- **索引**：细胞barcode（必须唯一）
- **常见列**：
  ```python
  adata.obs.columns
  # ['sample', 'batch', 'n_genes', 'total_counts', 
  #  'pct_counts_mt', 'doublet_score', 'leiden', 'cell_type']
  ```

### **3. var：基因注释数据**
```python
adata.var  # 变量（基因）的注释信息
```
- **类型**：`pandas.DataFrame`
- **形状**：`(n_var, n_var_columns)`
- **索引**：基因标识符（必须唯一）
- **常见列**：
  ```python
  adata.var.columns
  # ['gene_ids', 'gene_symbol', 'gene_name', 'feature_type',
  #  'n_cells', 'mean_counts', 'highly_variable']
  ```

## **扩展数据结构**

### **4. uns：非结构化注释**
```python
adata.uns  # 无结构化存储
```
- **类型**：嵌套的Python字典
- **用途**：存储不适合放在obs/var中的任意数据
- **常见内容**：
  ```python
  adata.uns.keys()
  # ['neighbors', 'pca', 'umap', 'leiden', 'rank_genes_groups', 'log1p']
  
  # 聚类信息
  adata.uns['leiden']
  # {'params': {'resolution': 1.0, 'random_state': 0},
  #  'connectivities': ...,
  #  'distances': ...}
  
  # 差异表达结果
  adata.uns['rank_genes_groups']
  # {'names': array([['CD3D', 'CD3E', ...]]),
  #  'scores': array([[30.5, 28.2, ...]]),
  #  'pvals': array([[1e-10, 1e-9, ...]])}
  ```

### **5. obsm：细胞多维嵌入**
```python
adata.obsm  # 细胞的多维坐标
```
- **类型**：`dict` of `numpy.ndarray`
- **形状**：每个数组为`(n_obs, n_components)`
- **常见内容**：
  ```python
  adata.obsm.keys()
  # ['X_pca', 'X_umap', 'X_tsne', 'X_diffmap']
  
  # PCA坐标
  print(adata.obsm['X_pca'].shape)  # (5000, 50) - 5000个细胞，50个PCs
  
  # UMAP坐标
  print(adata.obsm['X_umap'].shape)  # (5000, 2) - 5000个细胞，2个维度
  ```

### **6. varm：基因多维嵌入**
```python
adata.varm  # 基因的多维坐标
```
- **类型**：`dict` of `numpy.ndarray`
- **形状**：每个数组为`(n_var, n_components)`
- **用途**：存储基因级别的降维结果
  ```python
  # 例如：基因的PCA坐标（在基因共表达分析中）
  adata.varm['PCs']  # 形状：(20000, 50)
  ```

### **7. obsp：细胞间关系矩阵**
```python
adata.obsp  # 细胞-细胞关系
```
- **类型**：`dict` of 稀疏矩阵
- **形状**：每个矩阵为`(n_obs, n_obs)`
- **常见内容**：
  ```python
  adata.obsp.keys()
  # ['connectivities', 'distances', 'correlations']
  
  # 邻接矩阵（用于聚类）
  print(adata.obsp['connectivities'].shape)  # (5000, 5000)
  print(adata.obsp['distances'].shape)       # (5000, 5000)
  ```

### **8. varp：基因间关系矩阵**
```python
adata.varp  # 基因-基因关系
```
- **类型**：`dict` of 稀疏矩阵
- **形状**：每个矩阵为`(n_var, n_var)`
- **用途**：存储基因相关性、共表达网络等
  ```python
  adata.varp['correlations']  # 基因相关性矩阵
  ```

### **9. layers：多层面表达数据**
```python
adata.layers  # 附加表达矩阵
```
- **类型**：`dict` of 矩阵（与X同形状）
- **用途**：存储原始计数、归一化值、imputed值等
  ```python
  adata.layers.keys()
  # ['counts', 'normalized', 'log1p', 'scaled']
  
  # 原始计数
  adata.layers['counts'] = adata.X.copy()  # 保存原始计数
  adata.X = sc.pp.log1p(adata.X)           # 对X进行log转换
  
  # 访问不同层的数据
  raw_counts = adata.layers['counts']
  log_transformed = adata.X
  ```


